# Sistema RAG para análisis de mercado de Ethereum

Pipeline en orden de ejecución:

1. **Setup** — librerías, rutas, claves API, cliente Gemini
2. **Datos cripto + macro** — carga CSV, descarga macro (yfinance), correlaciones
3. **Funciones de cálculo** — RSI, volatilidad, snapshot completo
4. **ChromaDB** — inicialización + funciones de indexado y búsqueda
5. **Corpus histórico** — dataset Kaggle de noticias 2021-2023
6. **Noticias frescas** — RSS + NewsData.io con filtro de calidad
7. **Pipeline RAG** — carga TXT, snapshot a lenguaje natural, responder_pregunta
8. **Evaluación sistemática** — 20 preguntas en 4 bloques

Ejecuta en orden. Cada sección depende de las anteriores.

## 1. Setup

In [81]:
# Librerías y rutas centralizadas
import os
import re
import ast
import time
import json
import shutil
import hashlib
import requests
from datetime import datetime
from collections import Counter
from email.utils import parsedate_to_datetime

import torch
import pandas as pd
import chromadb
import kagglehub
import yfinance as yf
import feedparser
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(override=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}")

# Rutas
BASE_DIR          = r"C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF"
RUTA_CSV          = os.path.join(BASE_DIR, "data", "csv", "df_merged.csv")
RUTA_NOTICIAS_DIR = os.path.join(BASE_DIR, "data", "news")
RUTA_NOTICIAS_CSV = os.path.join(RUTA_NOTICIAS_DIR, "cryptonews.csv")
RUTA_TXT_GENERAL  = os.path.join(BASE_DIR, "data", "txt", "contexto_estatico_eth_v2.txt")
RUTA_TXT_ETH      = os.path.join(BASE_DIR, "data", "txt", "contexto_ethereum_v1.txt")
RUTA_CHROMA       = os.path.join(BASE_DIR, "chroma_db")
RUTA_RESULTADOS   = os.path.join(BASE_DIR, "data", "resultados_eval.json")

# Hiperparámetros del RAG
TOP_K_RETRIEVAL = 30   # cuántos docs recupera la búsqueda inicial
TOP_K_FINAL     = 5    # cuántos se inyectan al prompt final

# Modelo Gemini
MODELO_LLM       = "gemini-2.5-flash"
PAUSA_RATE_LIMIT = 13   # segundos entre llamadas (free tier ~5 RPM)

print("Setup OK ✓")

Dispositivo: cuda
Setup OK ✓


In [82]:
# Cliente Gemini + función de embeddings
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT"):
    """
    task_type:
      - 'RETRIEVAL_DOCUMENT' para indexar
      - 'RETRIEVAL_QUERY'    para buscar
    """
    if isinstance(texts, str):
        texts = [texts]
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=texts,
        config=types.EmbedContentConfig(task_type=task_type),
    )
    return [e.values for e in result.embeddings]

# Test rápido
test = embed_texts(["prueba"], task_type="RETRIEVAL_QUERY")
print(f"Cliente Gemini OK ✓  |  Dimensiones embedding: {len(test[0])}")

Cliente Gemini OK ✓  |  Dimensiones embedding: 3072


In [83]:
# Claves API de noticias frescas
NEWSDATA_KEY = os.environ.get("NEWSDATA_KEY")

if NEWSDATA_KEY:
    print(f"NewsData.io key cargada ✓ ({NEWSDATA_KEY[:10]}...)")
else:
    print("⚠️  Falta NEWSDATA_KEY en .env (las noticias en vivo no funcionarán)")

NewsData.io key cargada ✓ (pub_fd0a05...)


## 2. Datos cripto históricos + macro

Carga el CSV de cripto (BTC/ETH OHLCV + dominancias + fear&greed + inflación + tipos),
descarga datos macro de yfinance (DXY, oro, Nasdaq, US10Y, USD/JPY) y calcula correlaciones rolling.

In [84]:
# Cargar CSV cripto
df = pd.read_csv(RUTA_CSV, parse_dates=["date"], index_col="date")
df.sort_index(inplace=True)

print(f"Filas: {len(df)}  |  Rango: {df.index.min().date()} → {df.index.max().date()}")
print(f"Columnas ({len(df.columns)}): {df.columns.tolist()}")
df.head(3)

Filas: 3018  |  Rango: 2018-02-01 → 2026-05-07
Columnas (19): ['btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume', 'eth_open', 'eth_high', 'eth_low', 'eth_close', 'eth_volume', 'btc_dominance', 'eth_dominance', 'alt_dominance', 'fear_greed', 'FearGreed_Label', 'inflation', 'btc_mcap', 'eth_mcap', 'fed_rate']


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2018-02-01,10237.299805,10288.799805,8812.280273,9170.540039,9.959400e+09,1119.369995,1161.349976,984.818970,1036.790039,5.261680e+09,0.334251,0.211308,0.454441,30.0,Fear,2.263469,1.703042e+11,1.076635e+11,1.42
2018-02-02,9142.280273,9142.280273,7796.490234,8830.750000,1.272690e+10,1035.770020,1035.770020,757.979980,915.784973,6.713290e+09,0.339622,0.221066,0.439312,15.0,Extreme Fear,2.263469,1.527442e+11,9.942376e+10,1.42
2018-02-03,8852.120117,9430.750000,8251.629883,9174.910156,7.263790e+09,919.210999,991.942993,847.690002,964.018982,3.243480e+09,0.350252,0.209758,0.439990,40.0,Fear,2.263469,1.487152e+11,8.906186e+10,1.42


In [85]:
# Descargar datos macro vía yfinance
TICKERS_MACRO = {
    "dxy":     "DX-Y.NYB",   # Indice dólar
    "oro":     "GC=F",       # Oro futuros
    "nasdaq":  "^NDX",       # Nasdaq 100
    "us10y":   "^TNX",       # Yield bono 10 años USA
    "usdjpy":  "JPY=X",      # USD/JPY
}

def descargar_macro(tickers, start_date, end_date):
    dfs = {}
    for nombre, ticker in tickers.items():
        print(f"  Descargando {nombre} ({ticker})...")
        try:
            data = yf.download(ticker, start=start_date, end=end_date,
                               progress=False, auto_adjust=True)
            if data.empty:
                print(f"    ⚠️ Sin datos para {ticker}")
                continue
            dfs[nombre] = data["Close"]
        except Exception as e:
            print(f"    ❌ Error en {ticker}: {e}")
    df_macro = pd.concat(dfs, axis=1)
    df_macro.columns = list(dfs.keys())
    df_macro.index = pd.to_datetime(df_macro.index).tz_localize(None)
    return df_macro

fecha_inicio = df.index.min().strftime("%Y-%m-%d")
fecha_fin    = (df.index.max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

print(f"Descargando macro desde {fecha_inicio} hasta {fecha_fin}...\n")
df_macro = descargar_macro(TICKERS_MACRO, fecha_inicio, fecha_fin)
print(f"\nMacro descargado: {df_macro.shape}")
df_macro.tail(3)

Descargando macro desde 2018-02-01 hasta 2026-05-08...

  Descargando dxy (DX-Y.NYB)...
  Descargando oro (GC=F)...
  Descargando nasdaq (^NDX)...
  Descargando us10y (^TNX)...
  Descargando usdjpy (JPY=X)...

Macro descargado: (2152, 5)


,dxy,oro,nasdaq,us10y,usdjpy
Date,,,,,
2026-05-05,98.480003,4555.799805,28015.060547,4.416,157.194000
2026-05-06,98.019997,4681.899902,28599.169922,4.356,157.677002
2026-05-07,97.973999,4726.899902,28651.509766,4.376,156.574997


In [86]:
# Mergear cripto + macro y calcular variables derivadas
df_completo = df.join(df_macro, how="left").ffill()

# Cambios porcentuales 30 días
df_completo["dxy_chg_30d"]    = df_completo["dxy"].pct_change(30) * 100
df_completo["nasdaq_chg_30d"] = df_completo["nasdaq"].pct_change(30) * 100
df_completo["oro_chg_30d"]    = df_completo["oro"].pct_change(30) * 100
df_completo["us10y_chg_30d"]  = df_completo["us10y"].diff(30)

# Correlaciones rolling 30 días BTC vs activos macro
df_completo["btc_ret"]    = df_completo["btc_close"].pct_change()
df_completo["dxy_ret"]    = df_completo["dxy"].pct_change()
df_completo["nasdaq_ret"] = df_completo["nasdaq"].pct_change()
df_completo["oro_ret"]    = df_completo["oro"].pct_change()

df_completo["corr_btc_dxy_30d"]    = df_completo["btc_ret"].rolling(30).corr(df_completo["dxy_ret"])
df_completo["corr_btc_nasdaq_30d"] = df_completo["btc_ret"].rolling(30).corr(df_completo["nasdaq_ret"])
df_completo["corr_btc_oro_30d"]    = df_completo["btc_ret"].rolling(30).corr(df_completo["oro_ret"])

print(f"DataFrame completo: {df_completo.shape}")
print(f"Columnas nuevas añadidas:")
nuevas = [c for c in df_completo.columns if c not in df.columns]
for c in nuevas:
    print(f"  - {c}")

DataFrame completo: (3018, 35)
Columnas nuevas añadidas:
  - dxy
  - oro
  - nasdaq
  - us10y
  - usdjpy
  - dxy_chg_30d
  - nasdaq_chg_30d
  - oro_chg_30d
  - us10y_chg_30d
  - btc_ret
  - dxy_ret
  - nasdaq_ret
  - oro_ret
  - corr_btc_dxy_30d
  - corr_btc_nasdaq_30d
  - corr_btc_oro_30d


## 3. Funciones de cálculo (indicadores técnicos + snapshot)

Indicadores básicos (RSI, volatilidad, distribución sentimiento) y la función `calcular_snapshot_completo()`
que devuelve el dict con todos los datos del día listos para inyectar al prompt.

In [87]:
# Indicadores técnicos
def calcular_rsi(serie, periodo=14):
    delta = serie.diff()
    ganancias = delta.where(delta > 0, 0).rolling(periodo).mean()
    perdidas  = -delta.where(delta < 0, 0).rolling(periodo).mean()
    rs = ganancias / perdidas
    return 100 - (100 / (1 + rs))

def calcular_volatilidad(serie, ventana):
    """Volatilidad realizada anualizada en %."""
    return serie.pct_change().rolling(ventana).std() * (365 ** 0.5) * 100

def distribucion_fg_14d(df, fecha):
    ultimos_14 = df.loc[:fecha, "fear_greed"].iloc[-14:]
    return {
        "miedo_extremo":   int((ultimos_14 < 25).sum()),
        "miedo":           int(((ultimos_14 >= 25) & (ultimos_14 < 45)).sum()),
        "neutral":         int(((ultimos_14 >= 45) & (ultimos_14 < 55)).sum()),
        "codicia":         int(((ultimos_14 >= 55) & (ultimos_14 < 75)).sum()),
        "codicia_extrema": int((ultimos_14 >= 75).sum()),
    }

def fecha_ath(df, columna, hasta):
    serie = df.loc[:hasta, columna]
    return serie.idxmax(), serie.max()

print("✓ Funciones de indicadores técnicos cargadas")

✓ Funciones de indicadores técnicos cargadas


In [88]:
def calcular_snapshot_completo(df, fecha=None):
    """
    Snapshot completo: cripto + macro + indicadores + comparativas temporales.
    Devuelve dict con todos los datos del día listos para inyectar al prompt.
    """
    if fecha is None:
        row = df.iloc[-1]
        fecha = df.index[-1]
    else:
        row = df.loc[fecha]
    
    fecha_ts = pd.Timestamp(fecha)
    df_pasado = df.loc[:fecha]
    
    # ─── Helpers ──────────────────────────────────────────────────────
    def hace_n_dias(columna, n):
        if columna not in df_pasado.columns or len(df_pasado) < n + 1:
            return None
        v = df_pasado[columna].iloc[-(n+1)]
        return float(v) if pd.notna(v) else None
    
    def delta_pct(actual, pasado):
        if actual is None or pasado is None or pasado == 0:
            return None
        return round(((actual / pasado) - 1) * 100, 2)
    
    def delta_pp(actual_pct, pasado_pct):
        if actual_pct is None or pasado_pct is None:
            return None
        return round(actual_pct - pasado_pct, 2)
    
    # ─── Drawdown desde ATH ───────────────────────────────────────────
    eth_ath = df_pasado["eth_close"].max()
    btc_ath = df_pasado["btc_close"].max()
    eth_dd  = (row["eth_close"] / eth_ath) - 1
    btc_dd  = (row["btc_close"] / btc_ath) - 1
    
    # ─── Halvings ─────────────────────────────────────────────────────
    halvings = [pd.Timestamp("2012-11-28"), pd.Timestamp("2016-07-09"),
                pd.Timestamp("2020-05-11"), pd.Timestamp("2024-04-19")]
    halvings_pasados = [h for h in halvings if h <= fecha_ts]
    dias_post_halving = (fecha_ts - halvings_pasados[-1]).days if halvings_pasados else None
    
    # ─── Racha miedo extremo ──────────────────────────────────────────
    fg_serie = df.loc[:fecha, "fear_greed"]
    fg_bajo = (fg_serie < 25).astype(int)
    racha_miedo = 0
    for v in fg_bajo.iloc[::-1]:
        if v == 1:
            racha_miedo += 1
        else:
            break
    
    # ─── Indicadores técnicos ─────────────────────────────────────────
    rsi_eth = calcular_rsi(df_pasado["eth_close"], 14)
    rsi_btc = calcular_rsi(df_pasado["btc_close"], 14)
    
    vol_eth_7d  = calcular_volatilidad(df_pasado["eth_close"], 7)
    vol_eth_30d = calcular_volatilidad(df_pasado["eth_close"], 30)
    vol_eth_90d = calcular_volatilidad(df_pasado["eth_close"], 90)
    vol_btc_30d = calcular_volatilidad(df_pasado["btc_close"], 30)
    
    # ─── Retornos ─────────────────────────────────────────────────────
    retorno_eth_7d  = delta_pct(row["eth_close"], hace_n_dias("eth_close", 7))
    retorno_eth_30d = delta_pct(row["eth_close"], hace_n_dias("eth_close", 30))
    retorno_btc_7d  = delta_pct(row["btc_close"], hace_n_dias("btc_close", 7))
    retorno_btc_30d = delta_pct(row["btc_close"], hace_n_dias("btc_close", 30))
    
    # ─── ATH histórico vs ciclo actual ────────────────────────────────
    ath_eth_fecha, ath_eth_precio = fecha_ath(df, "eth_close", fecha)
    ath_btc_fecha, ath_btc_precio = fecha_ath(df, "btc_close", fecha)
    
    if halvings_pasados:
        ultimo_halving = halvings_pasados[-1]
        df_ciclo = df.loc[ultimo_halving:fecha]
        ath_eth_ciclo_precio = df_ciclo["eth_close"].max()
        ath_btc_ciclo_precio = df_ciclo["btc_close"].max()
        ath_eth_ciclo_supera = ath_eth_ciclo_precio > ath_eth_precio
        ath_btc_ciclo_supera = ath_btc_ciclo_precio > ath_btc_precio
    else:
        ath_eth_ciclo_precio = ath_btc_ciclo_precio = None
        ath_eth_ciclo_supera = ath_btc_ciclo_supera = None
    
    # ─── Distribución sentimiento 14d y 30d ───────────────────────────
    fg_dist_14 = distribucion_fg_14d(df, fecha)
    
    ultimos_30 = df.loc[:fecha, "fear_greed"].iloc[-30:]
    fg_dist_30 = {
        "miedo_extremo":   int((ultimos_30 < 25).sum()),
        "miedo":           int(((ultimos_30 >= 25) & (ultimos_30 < 45)).sum()),
        "neutral":         int(((ultimos_30 >= 45) & (ultimos_30 < 55)).sum()),
        "codicia":         int(((ultimos_30 >= 55) & (ultimos_30 < 75)).sum()),
        "codicia_extrema": int((ultimos_30 >= 75).sum()),
    }
    
    # Media F&G mes actual vs mes anterior
    media_fg_30d = float(df.loc[:fecha, "fear_greed"].iloc[-30:].mean())
    if len(df.loc[:fecha]) >= 60:
        media_fg_30_60d = float(df.loc[:fecha, "fear_greed"].iloc[-60:-30].mean())
    else:
        media_fg_30_60d = None
    
    # ─── Dominancias hace 7/30/90 días ────────────────────────────────
    dom_btc_actual = round(row["btc_dominance"] * 100, 2)
    dom_eth_actual = round(row["eth_dominance"] * 100, 2)
    
    dom_btc_7d  = round(hace_n_dias("btc_dominance", 7)  * 100, 2) if hace_n_dias("btc_dominance", 7)  else None
    dom_btc_30d = round(hace_n_dias("btc_dominance", 30) * 100, 2) if hace_n_dias("btc_dominance", 30) else None
    dom_btc_90d = round(hace_n_dias("btc_dominance", 90) * 100, 2) if hace_n_dias("btc_dominance", 90) else None
    
    dom_eth_7d  = round(hace_n_dias("eth_dominance", 7)  * 100, 2) if hace_n_dias("eth_dominance", 7)  else None
    dom_eth_30d = round(hace_n_dias("eth_dominance", 30) * 100, 2) if hace_n_dias("eth_dominance", 30) else None
    dom_eth_90d = round(hace_n_dias("eth_dominance", 90) * 100, 2) if hace_n_dias("eth_dominance", 90) else None
    
    # ─── Macro hace 30 días ───────────────────────────────────────────
    dxy_actual    = float(row.get("dxy"))    if pd.notna(row.get("dxy"))    else None
    nasdaq_actual = float(row.get("nasdaq")) if pd.notna(row.get("nasdaq")) else None
    oro_actual    = float(row.get("oro"))    if pd.notna(row.get("oro"))    else None
    us10y_actual  = float(row.get("us10y"))  if pd.notna(row.get("us10y"))  else None
    
    dxy_30d_atras    = hace_n_dias("dxy", 30)
    nasdaq_30d_atras = hace_n_dias("nasdaq", 30)
    oro_30d_atras    = hace_n_dias("oro", 30)
    us10y_30d_atras  = hace_n_dias("us10y", 30)
    
    # ─── Percentil volatilidad año ────────────────────────────────────
    vol_actual = vol_eth_30d.iloc[-1] if pd.notna(vol_eth_30d.iloc[-1]) else None
    vol_ultimo_ano = vol_eth_30d.iloc[-365:].dropna() if len(vol_eth_30d) >= 365 else vol_eth_30d.dropna()
    if vol_actual is not None and len(vol_ultimo_ano) > 0:
        vol_percentil = round((vol_ultimo_ano < vol_actual).sum() / len(vol_ultimo_ano) * 100, 1)
    else:
        vol_percentil = None
    
    # ─── Drawdown último mes ──────────────────────────────────────────
    eth_max_30d = df_pasado["eth_close"].iloc[-30:].max() if len(df_pasado) >= 30 else None
    btc_max_30d = df_pasado["btc_close"].iloc[-30:].max() if len(df_pasado) >= 30 else None
    dd_eth_mes_pct = round(((row["eth_close"] / eth_max_30d) - 1) * 100, 2) if eth_max_30d else None
    dd_btc_mes_pct = round(((row["btc_close"] / btc_max_30d) - 1) * 100, 2) if btc_max_30d else None
    
    return {
        "fecha":                 str(fecha_ts.date()),
        # Cripto
        "precio_eth_usd":        round(row["eth_close"], 2),
        "precio_btc_usd":        round(row["btc_close"], 2),
        "ratio_eth_btc":         round(row["eth_close"] / row["btc_close"], 5),
        "drawdown_eth_pct":      round(eth_dd * 100, 1),
        "drawdown_btc_pct":      round(btc_dd * 100, 1),
        "drawdown_eth_ultimo_mes_pct": dd_eth_mes_pct,
        "drawdown_btc_ultimo_mes_pct": dd_btc_mes_pct,
        # Dominancias
        "dominancia_btc_pct":         dom_btc_actual,
        "dominancia_eth_pct":         dom_eth_actual,
        "dominancia_btc_hace_7d":     dom_btc_7d,
        "dominancia_btc_hace_30d":    dom_btc_30d,
        "dominancia_btc_hace_90d":    dom_btc_90d,
        "dominancia_eth_hace_7d":     dom_eth_7d,
        "dominancia_eth_hace_30d":    dom_eth_30d,
        "dominancia_eth_hace_90d":    dom_eth_90d,
        "delta_dom_btc_30d_pp":       delta_pp(dom_btc_actual, dom_btc_30d),
        "delta_dom_eth_30d_pp":       delta_pp(dom_eth_actual, dom_eth_30d),
        # Sentimiento
        "miedo_codicia":              int(row["fear_greed"]),
        "miedo_codicia_etiqueta":     row.get("FearGreed_Label", ""),
        "dias_consecutivos_miedo_extremo": racha_miedo,
        "dias_desde_ultimo_halving":  dias_post_halving,
        "media_fg_ultimo_mes":        round(media_fg_30d, 1),
        "media_fg_mes_anterior":      round(media_fg_30_60d, 1) if media_fg_30_60d else None,
        "fg_dist_14d_miedo_extremo":  fg_dist_14["miedo_extremo"],
        "fg_dist_14d_miedo":          fg_dist_14["miedo"],
        "fg_dist_14d_neutral":        fg_dist_14["neutral"],
        "fg_dist_14d_codicia":        fg_dist_14["codicia"],
        "fg_dist_14d_codicia_extrema":fg_dist_14["codicia_extrema"],
        "fg_dist_30d_miedo_extremo":  fg_dist_30["miedo_extremo"],
        "fg_dist_30d_miedo":          fg_dist_30["miedo"],
        "fg_dist_30d_neutral":        fg_dist_30["neutral"],
        "fg_dist_30d_codicia":        fg_dist_30["codicia"],
        "fg_dist_30d_codicia_extrema":fg_dist_30["codicia_extrema"],
        # Macro
        "indice_dolar_dxy":      round(dxy_actual, 2)    if dxy_actual    else None,
        "oro_usd":               round(oro_actual, 2)    if oro_actual    else None,
        "nasdaq_100":            round(nasdaq_actual, 2) if nasdaq_actual else None,
        "yield_bono_10y_pct":    round(us10y_actual, 2)  if us10y_actual  else None,
        "usd_jpy":               round(row.get("usdjpy", 0), 2) if pd.notna(row.get("usdjpy")) else None,
        "tipos_fed_pct":         round(row["fed_rate"], 2),
        "inflacion_pct":         round(row["inflation"], 2),
        "dxy_cambio_30d_pct":    delta_pct(dxy_actual, dxy_30d_atras),
        "nasdaq_cambio_30d_pct": delta_pct(nasdaq_actual, nasdaq_30d_atras),
        "oro_cambio_30d_pct":    delta_pct(oro_actual, oro_30d_atras),
        "us10y_cambio_30d_pp":   delta_pp(us10y_actual, us10y_30d_atras),
        # Correlaciones
        "corr_btc_dxy_30d":      round(row.get("corr_btc_dxy_30d", 0), 2)    if pd.notna(row.get("corr_btc_dxy_30d"))    else None,
        "corr_btc_nasdaq_30d":   round(row.get("corr_btc_nasdaq_30d", 0), 2) if pd.notna(row.get("corr_btc_nasdaq_30d")) else None,
        "corr_btc_oro_30d":      round(row.get("corr_btc_oro_30d", 0), 2)    if pd.notna(row.get("corr_btc_oro_30d"))    else None,
        # Indicadores
        "rsi_eth_14d":           round(rsi_eth.iloc[-1], 2) if pd.notna(rsi_eth.iloc[-1]) else None,
        "rsi_btc_14d":           round(rsi_btc.iloc[-1], 2) if pd.notna(rsi_btc.iloc[-1]) else None,
        "volatilidad_eth_7d":    round(vol_eth_7d.iloc[-1], 2)  if pd.notna(vol_eth_7d.iloc[-1])  else None,
        "volatilidad_eth_30d":   round(vol_eth_30d.iloc[-1], 2) if pd.notna(vol_eth_30d.iloc[-1]) else None,
        "volatilidad_eth_90d":   round(vol_eth_90d.iloc[-1], 2) if pd.notna(vol_eth_90d.iloc[-1]) else None,
        "volatilidad_btc_30d":   round(vol_btc_30d.iloc[-1], 2) if pd.notna(vol_btc_30d.iloc[-1]) else None,
        "volatilidad_eth_percentil_1y": vol_percentil,
        # Retornos
        "retorno_eth_7d":        retorno_eth_7d,
        "retorno_eth_30d":       retorno_eth_30d,
        "retorno_btc_7d":        retorno_btc_7d,
        "retorno_btc_30d":       retorno_btc_30d,
        # ATH
        "ath_eth_precio":        round(ath_eth_precio, 2),
        "ath_eth_fecha":         str(ath_eth_fecha.date()),
        "dias_desde_ath_eth":    (fecha_ts - ath_eth_fecha).days,
        "ath_btc_precio":        round(ath_btc_precio, 2),
        "ath_btc_fecha":         str(ath_btc_fecha.date()),
        "dias_desde_ath_btc":    (fecha_ts - ath_btc_fecha).days,
        "ath_eth_ciclo_actual_precio":   round(ath_eth_ciclo_precio, 2) if ath_eth_ciclo_precio else None,
        "ath_eth_ciclo_supera_anterior": ath_eth_ciclo_supera,
        "ath_btc_ciclo_actual_precio":   round(ath_btc_ciclo_precio, 2) if ath_btc_ciclo_precio else None,
        "ath_btc_ciclo_supera_anterior": ath_btc_ciclo_supera,
    }


# Test del snapshot
snap = calcular_snapshot_completo(df_completo)
print(f"Snapshot del último día ({snap['fecha']}):")
for k, v in snap.items():
    print(f"  {k:38s}: {v}")

Snapshot del último día (2026-05-07):
  fecha                                 : 2026-05-07
  precio_eth_usd                        : 2295.94
  precio_btc_usd                        : 80158.78
  ratio_eth_btc                         : 0.02864
  drawdown_eth_pct                      : -52.5
  drawdown_btc_pct                      : -35.7
  drawdown_eth_ultimo_mes_pct           : -5.17
  drawdown_btc_ultimo_mes_pct           : -1.99
  dominancia_btc_pct                    : 58.49
  dominancia_eth_pct                    : 10.09
  dominancia_btc_hace_7d                : 58.02
  dominancia_btc_hace_30d               : 56.65
  dominancia_btc_hace_90d               : 56.5
  dominancia_eth_hace_7d                : 10.4
  dominancia_eth_hace_30d               : 10.46
  dominancia_eth_hace_90d               : 9.95
  delta_dom_btc_30d_pp                  : 1.84
  delta_dom_eth_30d_pp                  : -0.37
  miedo_codicia                         : 47
  miedo_codicia_etiqueta                : Neu

## 4. ChromaDB (base vectorial)

Inicializa la BD persistente y define las funciones `add_documents()` y `search()`.

In [89]:
# Inicializar ChromaDB persistente
chroma_client = chromadb.PersistentClient(path=RUTA_CHROMA)
collection = chroma_client.get_or_create_collection(
    name="crypto_news",
    metadata={"hnsw:space": "cosine"}
)
print(f"ChromaDB OK ✓  |  Docs en colección: {collection.count()}")

ChromaDB OK ✓  |  Docs en colección: 555


In [90]:
def add_documents(docs, batch_size=20):
    """Indexa documentos en ChromaDB con metadato date_int para filtros numéricos."""
    if not docs:
        return
    total = len(docs)
    for inicio in range(0, total, batch_size):
        lote = docs[inicio:inicio + batch_size]
        textos    = [d["text"] for d in lote]
        ids       = [d["id"]   for d in lote]
        metadatos = [{
            "date":     d["date"],
            "date_int": int(d["date"].replace("-", "")),  # ← clave numérica para filtros
            "source":   d["source"],
            "category": d["category"],
        } for d in lote]
        try:
            embeddings = embed_texts(textos, task_type="RETRIEVAL_DOCUMENT")
        except Exception as e:
            print(f"  ⚠️ Error en lote {inicio//batch_size + 1}: {e}")
            time.sleep(30)
            try:
                embeddings = embed_texts(textos, task_type="RETRIEVAL_DOCUMENT")
            except Exception as e2:
                print(f"  ❌ Lote saltado: {e2}")
                continue
        collection.add(ids=ids, embeddings=embeddings,
                       documents=textos, metadatas=metadatos)
        procesados = min(inicio + batch_size, total)
        print(f"  Lote {inicio//batch_size + 1}: {procesados}/{total} indexados")
        time.sleep(1.5)
    print(f"✓ Total en BD: {collection.count()}")


def fecha_a_int(fecha_str):
    """Convierte 'YYYY-MM-DD' a int YYYYMMDD para filtros numéricos en ChromaDB."""
    if fecha_str is None:
        return None
    return int(fecha_str.replace("-", ""))


def construir_where(date_from=None, date_to=None, category=None):
    filtros = []

    if date_from:
        filtros.append({"date_int": {"$gte": fecha_a_int(date_from)}})
    if date_to:
        filtros.append({"date_int": {"$lte": fecha_a_int(date_to)}})
    if category:
        filtros.append({"category": category})

    if len(filtros) == 0:
        return None
    if len(filtros) == 1:
        return filtros[0]
    return {"$and": filtros}


def formatear_resultados_chroma(results):
    docs = []
    for text, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        docs.append({
            "text": text,
            "date": meta["date"],
            "source": meta["source"],
            "category": meta["category"],
            "similarity": round(1 - dist, 4),
        })
    return docs


def aplicar_score_recencia(docs, date_to=None):
    fecha_ref = pd.Timestamp(date_to) if date_to else pd.Timestamp.now()

    for d in docs:
        try:
            dias_atras = max(0, (fecha_ref - pd.Timestamp(d["date"])).days)
            recency = max(0, 1 - dias_atras / 7)
        except Exception:
            recency = 0

        d["score"] = 0.7 * d["similarity"] + 0.3 * recency

    return docs


def search(query, top_k_final=12, date_from=None, date_to=None, pesos=None):
    """
    Retrieval realmente balanceado por categoría.
    Busca por separado en eth, macro y sentiment.
    """

    if collection.count() == 0:
        return []

    if pesos is None:
        pesos = {
            "eth": 4,
            "macro": 4,
            "sentiment": 4,
        }

    query_vec = embed_texts([query], task_type="RETRIEVAL_QUERY")

    seleccionadas = []

    for cat, k in pesos.items():
        where = construir_where(date_from=date_from, date_to=date_to, category=cat)

        try:
            results = collection.query(
                query_embeddings=query_vec,
                n_results=min(k * 4, collection.count()),
                where=where,
                include=["documents", "metadatas", "distances"],
            )
        except Exception as e:
            print(f"Error buscando categoría {cat}: {e}")
            continue

        docs_cat = formatear_resultados_chroma(results)
        docs_cat = aplicar_score_recencia(docs_cat, date_to=date_to)
        docs_cat = sorted(docs_cat, key=lambda x: x["score"], reverse=True)

        seleccionadas.extend(docs_cat[:k])

    seleccionadas = sorted(seleccionadas, key=lambda x: x["score"], reverse=True)

    return seleccionadas[:top_k_final]

print("✓ search() actualizado: balanceado por categoría + recencia")

def indexar_noticias_nuevas(noticias):
    """Filtra duplicados y mete las noticias nuevas en ChromaDB."""
    if not noticias:
        print("No hay noticias para indexar.")
        return
    try:
        existentes = collection.get(ids=[d["id"] for d in noticias])
        ids_existentes = set(existentes.get("ids", []))
    except Exception:
        ids_existentes = set()
    nuevas = [d for d in noticias if d["id"] not in ids_existentes]
    print(f"Total descargadas: {len(noticias)}  |  Ya en BD: {len(noticias)-len(nuevas)}  |  Nuevas: {len(nuevas)}")
    if nuevas:
        add_documents(nuevas)
    print(f"✓ Total docs en colección: {collection.count()}")

print("Funciones add_documents / search / indexar_noticias_nuevas listas ✓")

def limpiar_noticias_antiguas(dias=5):
    """
    Borra de Chroma las noticias anteriores a la ventana móvil.
    Evita acumulación diaria.
    """

    fecha_limite = pd.Timestamp.now() - pd.Timedelta(days=dias)
    fecha_limite_int = int(fecha_limite.strftime("%Y%m%d"))

    try:
        collection.delete(
            where={"date_int": {"$lt": fecha_limite_int}}
        )
        print(f"✓ Eliminadas noticias anteriores a {fecha_limite.strftime('%Y-%m-%d')}")
        print(f"✓ Total actual en BD: {collection.count()}")
    except Exception as e:
        print(f"⚠️ Error limpiando noticias antiguas: {e}")

✓ search() actualizado: balanceado por categoría + recencia
Funciones add_documents / search / indexar_noticias_nuevas listas ✓


## 5. Corpus histórico (dataset Kaggle)

Dataset `oliviervha/crypto-news` con ~31.000 artículos cripto entre 2021-2023.
Cobertura: bear market 2022, recuperación 2023, Merge de Ethereum, FTX.

**Atención:** indexar 31k docs vía API Gemini tarda 40-90 minutos. Se hace una sola vez.

In [91]:
# Descargar dataset solo si no existe
if not os.path.exists(RUTA_NOTICIAS_CSV):
    os.makedirs(RUTA_NOTICIAS_DIR, exist_ok=True)
    print("Descargando dataset de Kaggle (primera vez)...")
    cache_path = kagglehub.dataset_download("oliviervha/crypto-news")
    for f in os.listdir(cache_path):
        shutil.copy(os.path.join(cache_path, f), os.path.join(RUTA_NOTICIAS_DIR, f))
    print(f"Descargado en: {RUTA_NOTICIAS_DIR}")
else:
    print(f"CSV ya existe: {RUTA_NOTICIAS_CSV}")

CSV ya existe: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\news\cryptonews.csv


In [92]:
# Cargar y procesar el CSV en docs
df_news = pd.read_csv(
    RUTA_NOTICIAS_CSV,
    header=0,
    names=["date", "sentiment", "source", "subject", "text", "title", "url"]
)
df_news = df_news[df_news["date"] != "date"].reset_index(drop=True)

def parse_sentiment(s):
    try:
        return ast.literal_eval(s).get("class", "neutral")
    except Exception:
        return "neutral"

df_news["sentiment_class"] = df_news["sentiment"].apply(parse_sentiment)
df_news["date_clean"] = pd.to_datetime(df_news["date"], errors="coerce").dt.strftime("%Y-%m-%d")
df_news = df_news.dropna(subset=["date_clean"]).reset_index(drop=True)

PALABRAS_MACRO = ["sec ", "regulation", "ban ", "law ", "government",
                  "federal reserve", "fed ", "sanction", "inflation",
                  "tax ", "policy", "court", "legal", "congress", "senate"]
PALABRAS_ETH   = ["ethereum", " eth ", "bitcoin", " btc ", "defi", "hack",
                  "upgrade", "fork", "mining", "blockchain", "nft", "wallet"]

def asignar_categoria(row):
    texto = f"{row['subject']} {row['title']} {row['text']}".lower()
    if any(w in texto for w in PALABRAS_MACRO):
        return "macro"
    if any(w in texto for w in PALABRAS_ETH):
        return "eth"
    return "sentiment"

df_news["category"] = df_news.apply(asignar_categoria, axis=1)

def to_docs(df):
    docs = []
    for _, row in df.iterrows():
        titulo = str(row["title"]) if pd.notna(row["title"]) else ""
        texto  = str(row["text"])  if pd.notna(row["text"])  else ""
        combined = f"[{row['sentiment_class'].upper()}] {titulo}. {texto}"[:500]
        if not combined.strip():
            continue
        doc_id = hashlib.md5(combined[:100].encode()).hexdigest()
        docs.append({
            "id":       doc_id,
            "text":     combined,
            "date":     row["date_clean"],
            "source":   str(row["source"]),
            "category": row["category"],
        })
    return list({d["id"]: d for d in docs}.values())

docs_historicos = to_docs(df_news)

print(f"Total documentos: {len(docs_historicos)}")
print(f"Rango fechas: {min(d['date'] for d in docs_historicos)} → {max(d['date'] for d in docs_historicos)}")
print("\nDistribución por categoría:")
for cat in ["macro", "eth", "sentiment"]:
    n = sum(1 for d in docs_historicos if d["category"] == cat)
    print(f"  {cat:10s}: {n:>6}")

Total documentos: 31007
Rango fechas: 2021-10-12 → 2023-12-19

Distribución por categoría:
  macro     :   4713
  eth       :  20221
  sentiment :   6073


In [93]:
# INDEXAR EL CORPUS HISTÓRICO en ChromaDB
# ⚠️ Solo ejecutar si la colección está vacía. Tarda 40-90 minutos.

if collection.count() == 0:
    print(f"Indexando {len(docs_historicos)} docs históricos...")
    print("Esto va a tardar bastante. Puedes parar y reanudar otro día.\n")
    # add_documents(docs_historicos)   # ← descomenta cuando quieras lanzarlo
    print("⚠️ Indexado COMENTADO. Descomenta la línea anterior para lanzarlo.")
else:
    print(f"✓ Colección ya tiene {collection.count()} docs. Saltando indexado histórico.")

✓ Colección ya tiene 555 docs. Saltando indexado histórico.


## 6. Noticias frescas (RSS + NewsData.io)

Para mantener el RAG actualizado con noticias del día. Combina:
- **RSS feeds** (CoinDesk, Cointelegraph, Decrypt, etc.) — sin key, sin límite, ~1-2 semanas atrás
- **NewsData.io** — últimas 48h, requiere key gratuita

Se filtra calidad antes de indexar (descarta price predictions, shitcoin promos, etc.).

In [94]:
# Filtro de calidad (compartido entre RSS y NewsData)
def filtrar_noticias_calidad(noticias):
    """Descarta noticias de baja calidad (price predictions, promos, etc.)."""
    fuentes_blacklist = {
        "openpr", "techbullion", "equity_insider", "globenewswire",
        "prnewswire", "businesswire", "newsbtc",
    }
    palabras_spam = [
        "price prediction", "huge gains", "moonshot", "100x",
        "before listing", "presale", "millionaire",
        "to the moon", "next big thing", "10x potential",
    ]
    filtradas = []
    descartadas = 0
    for n in noticias:
        texto_lower = n["text"].lower()
        fuente_lower = n["source"].lower()
        if fuente_lower in fuentes_blacklist:
            descartadas += 1; continue
        if any(spam in texto_lower for spam in palabras_spam):
            descartadas += 1; continue
        memecoins = ["pepeto", "shiba", "dogecoin", "pepe", "bonk", "wif"]
        if sum(1 for m in memecoins if m in texto_lower) >= 1 and "bitcoin" not in texto_lower:
            descartadas += 1; continue
        filtradas.append(n)
    print(f"Filtradas: {len(filtradas)}  |  Descartadas: {descartadas}")
    return filtradas

print("✓ Filtro de calidad cargado")

✓ Filtro de calidad cargado


In [95]:

RSS_FEEDS = {
    # ─── CRIPTO (validadas) ──────────────────────────────────────────
    "CoinDesk":         "https://www.coindesk.com/arc/outboundfeeds/rss/",
    "Cointelegraph":    "https://cointelegraph.com/rss",
    "Decrypt":          "https://decrypt.co/feed",
    "CryptoSlate":      "https://cryptoslate.com/feed/",
    "Bitcoinist":       "https://bitcoinist.com/feed/",
    "UToday":           "https://u.today/rss",
    "CoinJournal":      "https://coinjournal.net/news/feed/",
    "BitcoinMagazine":  "https://bitcoinmagazine.com/feed",  # ← endpoint nuevo
    
    # ─── MACRO / FINANZAS ────────────────────────────────────────────
    "YahooFinance":     "https://finance.yahoo.com/news/rssindex",
    "MarketWatch":      "https://feeds.content.dowjones.io/public/rss/mw_topstories",
    "CNBCWorld":        "https://www.cnbc.com/id/100727362/device/rss/rss.html",
    "Investing":        "https://www.investing.com/rss/news_25.rss",
}


def descargar_rss_feeds(max_por_feed=30, max_dias_antiguedad=7):
    """
    Descarga RSS feeds. Filtra artículos más viejos que max_dias_antiguedad.
    """
    docs = []
    feeds_ok = 0
    feeds_fallidos = []
    
    for nombre, url in RSS_FEEDS.items():
        try:
            feed = feedparser.parse(url, request_headers={"User-Agent": "Mozilla/5.0"})
            if feed.bozo and not feed.entries:
                feeds_fallidos.append(nombre)
                print(f"  ✗ {nombre}: sin entradas")
                continue
            if not feed.entries:
                feeds_fallidos.append(nombre)
                print(f"  ✗ {nombre}: vacío")
                continue
        except Exception as e:
            feeds_fallidos.append(nombre)
            print(f"  ✗ {nombre}: {e}")
            continue
        
        contador_feed = 0
        ahora = pd.Timestamp.now()
        
        for entry in feed.entries[:max_por_feed]:
            titulo = entry.get("title", "") or ""
            resumen = entry.get("summary", "") or entry.get("description", "") or ""
            resumen = re.sub(r'<[^>]+>', '', resumen)
            resumen = re.sub(r'\s+', ' ', resumen).strip()
            
            fecha_raw = entry.get("published", "") or entry.get("updated", "")
            try:
                fecha_dt = parsedate_to_datetime(fecha_raw)
                fecha = fecha_dt.strftime("%Y-%m-%d")
                # Filtrar artículos viejos
                dias_atras = (ahora - pd.Timestamp(fecha_dt).tz_localize(None)).days
                if dias_atras > max_dias_antiguedad:
                    continue
            except Exception:
                fecha = ahora.strftime("%Y-%m-%d")
            
            texto = f"{titulo}. {resumen}"[:800].strip()
            if not texto or not titulo:
                continue
            
            texto_lower = (titulo + " " + resumen).lower()
            if any(k in texto_lower for k in ["fed", "powell", "trump", "tariff",
                                                  "inflation", "interest rate", "treasury",
                                                  "regulation", "sec ", "federal reserve",
                                                  "war", "ukraine", "china", "iran"]):
                categoria = "macro"

            elif any(k in texto_lower for k in ["bitcoin", "btc", "ethereum", "eth",
                                                "crypto", "blockchain", "defi", "stablecoin"]):
                categoria = "eth"
            
            else:
                categoria = "sentiment"
            
            doc_id = hashlib.md5(texto[:100].encode()).hexdigest()
            docs.append({
                "id": doc_id, "text": texto[:500], "date": fecha,
                "source": nombre, "category": categoria,
            })
            contador_feed += 1
        
        if contador_feed > 0:
            feeds_ok += 1
            print(f"  ✓ {nombre}: {contador_feed} artículos válidos")
    
    docs = list({d["id"]: d for d in docs}.values())
    print(f"\n✓ Total: {len(docs)} artículos | Feeds OK: {feeds_ok}/{len(RSS_FEEDS)}")
    if feeds_fallidos:
        print(f"  Feeds fallidos: {feeds_fallidos}")
    return docs


print("✓ RSS_FEEDS ampliado y descarga con filtro temporal cargada")

✓ RSS_FEEDS ampliado y descarga con filtro temporal cargada


In [96]:
# NewsData.io (últimas 48h, requiere key)
def descargar_noticias_newsdata(query="bitcoin OR ethereum OR crypto", paginas=2):
    """Descarga noticias de NewsData.io (free tier)."""
    if not NEWSDATA_KEY:
        print("⚠️  No hay key de NewsData")
        return []
    
    base_url = "https://newsdata.io/api/1/latest"
    docs = []
    next_page = None
    
    for pag in range(paginas):
        params = {"apikey": NEWSDATA_KEY, "q": query, "language": "en"}
        if next_page:
            params["page"] = next_page
        try:
            r = requests.get(base_url, params=params, timeout=20)
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            print(f"Error NewsData página {pag+1}: {e}"); break
        
        if data.get("status") != "success":
            print(f"Status no OK: {data}"); break
        
        for art in data.get("results", []):
            titulo = art.get("title", "") or ""
            descripcion = art.get("description", "") or ""
            contenido = art.get("content", "") or ""
            fecha = (art.get("pubDate", "") or "")[:10]
            fuente = art.get("source_id", "unknown")
            
            texto = f"{titulo}. {descripcion}"
            if contenido and len(contenido) > 100:
                texto += f" {contenido[:600]}"
            texto = texto[:800].strip()
            if not texto or not titulo or titulo == "[Removed]":
                continue
            
            texto_lower = (titulo + " " + descripcion).lower()
            if any(k in texto_lower for k in ["fed", "powell", "trump", "tariff",
                                                  "inflation", "interest rate",
                                                  "treasury", "regulation"]):
                categoria = "macro"
            elif any(k in texto_lower for k in ["bitcoin", "btc", "ethereum", "eth",
                                                "crypto", "blockchain", "defi"]):
                categoria = "eth"

            else:
                categoria = "sentiment"
            
            doc_id = hashlib.md5(texto[:100].encode()).hexdigest()
            docs.append({
                "id": doc_id, "text": texto[:500],
                "date": fecha if fecha else pd.Timestamp.now().strftime("%Y-%m-%d"),
                "source": fuente, "category": categoria,
            })

        
        next_page = data.get("nextPage")
        if not next_page:
            break
        time.sleep(1)
    
    return list({d["id"]: d for d in docs}.values())

print("✓ Función descargar_noticias_newsdata cargada")

✓ Función descargar_noticias_newsdata cargada


In [97]:
# Pipeline diario completo: descarga RSS + NewsData, filtra e indexa
def actualizar_noticias_diarias():
    """Ejecutar 1 vez al día para mantener el RAG con noticias frescas."""
    print(f"\n{'='*60}")
    print(f"Actualización noticias - {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
    print('='*60)
    
    # 1. RSS (1-2 semanas atrás)
    print("\n[1/2] RSS feeds...")
    rss = descargar_rss_feeds(max_por_feed=30, max_dias_antiguedad=5)
    print(f"  Total RSS: {len(rss)}")
    
    # 2. NewsData (últimas 48h)
    print("\n[2/2] NewsData...")
    nd_crypto = descargar_noticias_newsdata("bitcoin OR ethereum OR crypto", paginas=2)
    time.sleep(1)
    nd_macro  = descargar_noticias_newsdata("trump OR fed OR inflation", paginas=2)
    nd_total = list({d["id"]: d for d in nd_crypto + nd_macro}.values())
    print(f"  Total NewsData: {len(nd_total)}")
    
    # 3. Juntar, filtrar, indexar
    todas = list({d["id"]: d for d in rss + nd_total}.values())
    print(f"\nTotal antes de filtrar: {len(todas)}")

    filtradas = filtrar_noticias_calidad(todas)

    # Primero indexas nuevas
    indexar_noticias_nuevas(filtradas)

    # Luego limpias todo lo que salga de la ventana móvil
    limpiar_noticias_antiguas(dias=5)

# Ejecutar manualmente cuando quieras actualizar:
# actualizar_noticias_diarias()
print("Pipeline diario listo. Ejecuta actualizar_noticias_diarias() cuando quieras.")

Pipeline diario listo. Ejecuta actualizar_noticias_diarias() cuando quieras.


In [98]:
actualizar_noticias_diarias() 


Actualización noticias - 2026-05-07 18:34

[1/2] RSS feeds...
  ✓ CoinDesk: 25 artículos válidos
  ✓ Cointelegraph: 30 artículos válidos
  ✓ Decrypt: 30 artículos válidos
  ✓ CryptoSlate: 10 artículos válidos
  ✓ Bitcoinist: 8 artículos válidos
  ✓ UToday: 30 artículos válidos
  ✓ CoinJournal: 9 artículos válidos
  ✗ BitcoinMagazine: sin entradas
  ✓ YahooFinance: 30 artículos válidos
  ✓ MarketWatch: 10 artículos válidos
  ✓ CNBCWorld: 30 artículos válidos
  ✓ Investing: 10 artículos válidos

✓ Total: 222 artículos | Feeds OK: 11/12
  Feeds fallidos: ['BitcoinMagazine']
  Total RSS: 222

[2/2] NewsData...
  Total NewsData: 36

Total antes de filtrar: 258
Filtradas: 250  |  Descartadas: 8
Total descargadas: 250  |  Ya en BD: 39  |  Nuevas: 211
  Lote 1: 20/211 indexados
  Lote 2: 40/211 indexados
  Lote 3: 60/211 indexados
  Lote 4: 80/211 indexados
  ⚠️ Error en lote 5: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan

In [99]:
todos = collection.get(include=["metadatas"])
from collections import Counter
print(Counter(m["category"] for m in todos["metadatas"]))

Counter({'sentiment': 324, 'eth': 293, 'macro': 126})


## 7. Pipeline RAG (cargar TXT + responder_pregunta)

Carga los TXT estáticos (patrones generales + Ethereum), define `snapshot_a_lenguaje_natural()`
que convierte el snapshot a texto, y la función final `responder_pregunta()`.

In [100]:
RUTA_TXT_MOTOR = os.path.join(BASE_DIR, "data", "txt", "motor_razonamiento_mercado_v1.txt")

# Cargar los dos TXT estáticos
with open(RUTA_TXT_GENERAL, "r", encoding="utf-8") as f:
    CONTEXTO_ESTATICO = f.read()

with open(RUTA_TXT_ETH, "r", encoding="utf-8") as f:
    CONTEXTO_ETHEREUM = f.read()

with open(RUTA_TXT_MOTOR, "r", encoding="utf-8") as f:
    MOTOR_RAZONAMIENTO = f.read()
    
print(f"TXT general:   {len(CONTEXTO_ESTATICO):,} caracteres")
print(f"TXT Ethereum:  {len(CONTEXTO_ETHEREUM):,} caracteres")
print(f"Motor razonamiento: {len(MOTOR_RAZONAMIENTO):,} caracteres")
print(f"Total contexto estático: ~{(len(CONTEXTO_ESTATICO)+len(CONTEXTO_ETHEREUM)+len(MOTOR_RAZONAMIENTO))//4:,} tokens aprox.")

TXT general:   33,048 caracteres
TXT Ethereum:  24,981 caracteres
Motor razonamiento: 38,655 caracteres
Total contexto estático: ~24,171 tokens aprox.


In [101]:
def snapshot_a_lenguaje_natural(snap):
    """Convierte el dict del snapshot en texto en castellano natural."""
    L = [f"Fecha del análisis: {snap['fecha']}", ""]
    
    # PRECIOS Y POSICIÓN EN EL CICLO
    L.append("PRECIOS Y POSICIÓN EN EL CICLO:")
    L.append(f"  Precio de Ethereum: {snap['precio_eth_usd']} dólares")
    L.append(f"  Precio de Bitcoin: {snap['precio_btc_usd']} dólares")
    L.append(f"  Ratio Ethereum/Bitcoin: {snap['ratio_eth_btc']}")
    L.append(f"  Caída de Ethereum desde su máximo histórico: {snap['drawdown_eth_pct']}%")
    L.append(f"  Caída de Bitcoin desde su máximo histórico: {snap['drawdown_btc_pct']}%")
    if snap.get('drawdown_eth_ultimo_mes_pct') is not None:
        L.append(f"  Caída de Ethereum desde su máximo del último mes: {snap['drawdown_eth_ultimo_mes_pct']}%")
    if snap.get('drawdown_btc_ultimo_mes_pct') is not None:
        L.append(f"  Caída de Bitcoin desde su máximo del último mes: {snap['drawdown_btc_ultimo_mes_pct']}%")
    if snap['dias_desde_ultimo_halving'] is not None:
        L.append(f"  Días desde el último halving de Bitcoin: {snap['dias_desde_ultimo_halving']}")
    
    # MÁXIMOS HISTÓRICOS
    L.append("")
    L.append("MÁXIMOS HISTÓRICOS:")
    L.append(f"  ATH histórico de Ethereum: {snap['ath_eth_precio']} USD el {snap['ath_eth_fecha']} (hace {snap['dias_desde_ath_eth']} días)")
    L.append(f"  ATH histórico de Bitcoin: {snap['ath_btc_precio']} USD el {snap['ath_btc_fecha']} (hace {snap['dias_desde_ath_btc']} días)")
    if snap.get('ath_eth_ciclo_actual_precio'):
        sup_eth = "SÍ" if snap['ath_eth_ciclo_supera_anterior'] else "NO"
        L.append(f"  Máximo de Ethereum en el ciclo actual (post halving): {snap['ath_eth_ciclo_actual_precio']} USD. ¿Ha superado el ATH histórico anterior? {sup_eth}")
    if snap.get('ath_btc_ciclo_actual_precio'):
        sup_btc = "SÍ" if snap['ath_btc_ciclo_supera_anterior'] else "NO"
        L.append(f"  Máximo de Bitcoin en el ciclo actual (post halving): {snap['ath_btc_ciclo_actual_precio']} USD. ¿Ha superado el ATH histórico anterior? {sup_btc}")
    
    # DOMINANCIAS
    L.append("")
    L.append("DOMINANCIAS DE MERCADO Y SU EVOLUCIÓN:")
    L.append(f"  Dominancia actual de Bitcoin: {snap['dominancia_btc_pct']}%")
    if snap.get('dominancia_btc_hace_7d')  is not None: L.append(f"    Hace 7 días: {snap['dominancia_btc_hace_7d']}%")
    if snap.get('dominancia_btc_hace_30d') is not None: L.append(f"    Hace 30 días: {snap['dominancia_btc_hace_30d']}% (cambio: {snap['delta_dom_btc_30d_pp']:+.2f} puntos porcentuales)")
    if snap.get('dominancia_btc_hace_90d') is not None: L.append(f"    Hace 90 días: {snap['dominancia_btc_hace_90d']}%")
    L.append(f"  Dominancia actual de Ethereum: {snap['dominancia_eth_pct']}%")
    if snap.get('dominancia_eth_hace_7d')  is not None: L.append(f"    Hace 7 días: {snap['dominancia_eth_hace_7d']}%")
    if snap.get('dominancia_eth_hace_30d') is not None: L.append(f"    Hace 30 días: {snap['dominancia_eth_hace_30d']}% (cambio: {snap['delta_dom_eth_30d_pp']:+.2f} puntos porcentuales)")
    if snap.get('dominancia_eth_hace_90d') is not None: L.append(f"    Hace 90 días: {snap['dominancia_eth_hace_90d']}%")
    
    # SENTIMIENTO
    L.append("")
    L.append("SENTIMIENTO DE MERCADO:")
    L.append(f"  Índice de miedo y codicia actual: {snap['miedo_codicia']} ({snap['miedo_codicia_etiqueta']})")
    L.append(f"  Días consecutivos en miedo extremo: {snap['dias_consecutivos_miedo_extremo']}")
    L.append(f"  Media del índice último mes: {snap['media_fg_ultimo_mes']}")
    if snap.get('media_fg_mes_anterior') is not None:
        L.append(f"  Media del índice mes anterior: {snap['media_fg_mes_anterior']}")
    L.append("  Distribución últimos 14 días:")
    L.append(f"    Miedo extremo: {snap['fg_dist_14d_miedo_extremo']} días | Miedo: {snap['fg_dist_14d_miedo']} | Neutral: {snap['fg_dist_14d_neutral']} | Codicia: {snap['fg_dist_14d_codicia']} | Codicia extrema: {snap['fg_dist_14d_codicia_extrema']}")
    L.append("  Distribución últimos 30 días:")
    L.append(f"    Miedo extremo: {snap['fg_dist_30d_miedo_extremo']} días | Miedo: {snap['fg_dist_30d_miedo']} | Neutral: {snap['fg_dist_30d_neutral']} | Codicia: {snap['fg_dist_30d_codicia']} | Codicia extrema: {snap['fg_dist_30d_codicia_extrema']}")
    
    # INDICADORES TÉCNICOS
    L.append("")
    L.append("INDICADORES TÉCNICOS:")
    if snap.get('rsi_eth_14d') is not None: L.append(f"  RSI de Ethereum (14 días): {snap['rsi_eth_14d']}")
    if snap.get('rsi_btc_14d') is not None: L.append(f"  RSI de Bitcoin (14 días): {snap['rsi_btc_14d']}")
    if snap.get('volatilidad_eth_7d')  is not None: L.append(f"  Volatilidad anualizada de Ethereum (7 días): {snap['volatilidad_eth_7d']}%")
    if snap.get('volatilidad_eth_30d') is not None: L.append(f"  Volatilidad anualizada de Ethereum (30 días): {snap['volatilidad_eth_30d']}%")
    if snap.get('volatilidad_eth_90d') is not None: L.append(f"  Volatilidad anualizada de Ethereum (90 días): {snap['volatilidad_eth_90d']}%")
    if snap.get('volatilidad_eth_percentil_1y') is not None:
        L.append(f"  Percentil de la volatilidad actual de ETH respecto al último año: {snap['volatilidad_eth_percentil_1y']}%")
    if snap.get('volatilidad_btc_30d') is not None: L.append(f"  Volatilidad anualizada de Bitcoin (30 días): {snap['volatilidad_btc_30d']}%")
    
    # RENDIMIENTOS
    L.append("")
    L.append("RENDIMIENTOS RECIENTES:")
    if snap.get('retorno_eth_7d')  is not None: L.append(f"  Ethereum última semana: {snap['retorno_eth_7d']:+.2f}%")
    if snap.get('retorno_eth_30d') is not None: L.append(f"  Ethereum último mes: {snap['retorno_eth_30d']:+.2f}%")
    if snap.get('retorno_btc_7d')  is not None: L.append(f"  Bitcoin última semana: {snap['retorno_btc_7d']:+.2f}%")
    if snap.get('retorno_btc_30d') is not None: L.append(f"  Bitcoin último mes: {snap['retorno_btc_30d']:+.2f}%")
    
    # MACRO
    L.append("")
    L.append("CONTEXTO MACRO-FINANCIERO:")
    if snap.get('indice_dolar_dxy') is not None:
        delta = f" (cambio último mes: {snap['dxy_cambio_30d_pct']:+.2f}%)" if snap.get('dxy_cambio_30d_pct') is not None else ""
        L.append(f"  Índice del dólar (DXY): {snap['indice_dolar_dxy']}{delta}")
    if snap.get('oro_usd') is not None:
        delta = f" (cambio último mes: {snap['oro_cambio_30d_pct']:+.2f}%)" if snap.get('oro_cambio_30d_pct') is not None else ""
        L.append(f"  Oro: {snap['oro_usd']} dólares por onza{delta}")
    if snap.get('nasdaq_100') is not None:
        delta = f" (cambio último mes: {snap['nasdaq_cambio_30d_pct']:+.2f}%)" if snap.get('nasdaq_cambio_30d_pct') is not None else ""
        L.append(f"  Nasdaq 100: {snap['nasdaq_100']}{delta}")
    if snap.get('yield_bono_10y_pct') is not None:
        delta = f" (cambio último mes: {snap['us10y_cambio_30d_pp']:+.2f} puntos porcentuales)" if snap.get('us10y_cambio_30d_pp') is not None else ""
        L.append(f"  Yield del bono americano a 10 años: {snap['yield_bono_10y_pct']}%{delta}")
    if snap.get('usd_jpy') is not None:
        L.append(f"  Tipo de cambio USD/JPY: {snap['usd_jpy']}")
    L.append(f"  Tipos de la Reserva Federal: {snap['tipos_fed_pct']}%")
    L.append(f"  Inflación interanual EEUU: {snap['inflacion_pct']}%")
    
    # CORRELACIONES
    if snap.get('corr_btc_nasdaq_30d') is not None:
        L.append("")
        L.append("CORRELACIONES RECIENTES (ventana 30 días):")
        L.append(f"  Bitcoin vs Nasdaq 100: {snap['corr_btc_nasdaq_30d']}")
        L.append(f"  Bitcoin vs índice del dólar: {snap['corr_btc_dxy_30d']}")
        L.append(f"  Bitcoin vs oro: {snap['corr_btc_oro_30d']}")
    
    return "\n".join(L)

print("✓ Función snapshot_a_lenguaje_natural cargada")

✓ Función snapshot_a_lenguaje_natural cargada


In [102]:
def construir_prompt(pregunta, snapshot=None, noticias=None):
    """Construye el prompt completo: TXT general + TXT Ethereum + snapshot + noticias + pregunta."""
    snap_txt = snapshot_a_lenguaje_natural(snapshot) if snapshot else "Sin datos del día disponibles."
    
    if noticias:
        not_txt = "\n".join(
            f"  [{n['date']}] [{n['category']}] {n['source']}: {n['text']}"
            for n in noticias
        )
    else:
        not_txt = "  Sin noticias recuperadas."
    
    return f"""Eres un analista experto en mercados de criptomonedas, especializado en Ethereum.

    <MOTOR_DE_RAZONAMIENTO>
    {MOTOR_RAZONAMIENTO}
    </MOTOR_DE_RAZONAMIENTO>

    <CONTEXTO_GENERAL>
    {CONTEXTO_ESTATICO}
    </CONTEXTO_GENERAL>

    <CONTEXTO_ESPECIFICO_ETHEREUM>
    {CONTEXTO_ETHEREUM}
    </CONTEXTO_ESPECIFICO_ETHEREUM>

    <DATOS_DEL_DIA>
    {snap_txt}
    </DATOS_DEL_DIA>

    <NOTICIAS_RELEVANTES>
    {not_txt}
    </NOTICIAS_RELEVANTES>

    <PREGUNTA>
    {pregunta}
    </PREGUNTA>
    """

def responder_pregunta(pregunta, fecha=None, usar_rag=True, top_k=12, ventana_dias=7):
    """
    Pipeline completo: snapshot + RAG (top_k noticias frescas) + Gemini.
    
    ventana_dias: solo recupera noticias de los últimos N días (por defecto 7).
    """
    snap = calcular_snapshot_completo(df_completo, fecha=fecha)
    noticias = []
    
    if usar_rag and collection.count() > 0:
        # Ventana temporal: últimos N días desde la fecha del análisis
        fecha_analisis = pd.Timestamp(snap["fecha"])
        date_from = (fecha_analisis - pd.Timedelta(days=ventana_dias)).strftime("%Y-%m-%d")
        # date_to: 2 días después del análisis para no perder noticias publicadas justo después
        date_to = (fecha_analisis + pd.Timedelta(days=2)).strftime("%Y-%m-%d")
        
        noticias = search(
            pregunta,
            top_k_final=top_k,
            date_from=date_from,
            date_to=date_to,
            pesos={
                "eth": 4,
                "macro": 4,
                "sentiment": 4,
            }
        )
    
    prompt = construir_prompt(pregunta, snapshot=snap, noticias=noticias)
    
    try:
        respuesta = client.models.generate_content(
            model=MODELO_LLM, contents=prompt
        ).text
    except Exception as e:
        respuesta = f"❌ ERROR: {e}"
    
    log = {
        "fecha_analisis": snap["fecha"],
        "pregunta": pregunta,
        "snapshot": snap,
        "noticias_recuperadas": [
            {"date": n["date"], "cat": n["category"], "src": n["source"],
             "sim": n["similarity"], "score": round(n["score"], 4),
             "text": n["text"][:120]}
            for n in noticias
        ],
        "respuesta": respuesta,
    }
    return respuesta, log


print("✓ responder_pregunta() actualizado: ventana 7d + 12 noticias balanceadas")

✓ responder_pregunta() actualizado: ventana 7d + 12 noticias balanceadas


In [108]:
respuesta, log = responder_pregunta(
    "Hablame de como es tu motor de razonamiento y dame tu visión del mercado actual de Ethereum, teniendo en cuenta las noticias más relevantes de la última semana sin hacer tampoco una explicación muy extensa."
)
print(respuesta)
print(f"\nNoticias usadas: {len(log['noticias_recuperadas'])}")
print("\nDistribución por categoría:")
from collections import Counter
print(Counter(n['cat'] for n in log['noticias_recuperadas']))

Como analista macro-cripto, mi motor de razonamiento está diseñado para proporcionar una visión estructurada y probabilística del mercado, integrando múltiples capas de información y mitigando sesgos. Funciona de la siguiente manera:

En primer lugar, establece una **identidad y rol claros**, actuando como un estratega institucional que prioriza el rigor cuantitativo y el lenguaje probabilístico. Los **principios no negociables** incluyen hablar siempre en castellano natural, no inventar datos (marcando [INFERENCIA] o [NO HAY DATO] cuando sea necesario), evitar predicciones de precio concretas y mantener una convicción condicional.

Un **glosario interno** traduce identificadores técnicos de patrones (P1, E2, C7) a nombres descriptivos y comprensibles para el usuario, asegurando una comunicación clara.

El pilar fundamental es la **separación explícita de tres horizontes temporales**: táctico (días/semanas), cíclico (meses/1-2 años) y estructural (años). Cada horizonte tiene sus propia

## 8. Evaluación sistemática (20 preguntas)

20 preguntas en 4 bloques para evaluar el pipeline.

**⚠️ Atención:** consume cuota de Gemini free tier (~5 RPM). Hay pausa entre llamadas.
Tarda unos 5 minutos.

In [24]:
preguntas = [
    # BLOQUE 1 — Hechos históricos verificables
    "¿En qué año ocurrió el halving de Bitcoin de 2020 y cuánto bajó la recompensa de bloque? ¿Qué efecto histórico tuvo en el precio en los 12 meses siguientes?",
    "¿Cuál fue el precio máximo histórico (ATH) de Ethereum en USD y en qué fecha exacta ocurrió?",
    "Durante el ciclo alcista de 2017, ¿cuál fue el rango de precio de Ethereum desde su mínimo hasta su máximo? ¿Cuánto tiempo duró aproximadamente ese movimiento?",
    "¿Cuántos halvings de Bitcoin han ocurrido hasta la fecha y cuáles son sus fechas aproximadas?",
    "¿En qué rango de precios cotizaba Bitcoin durante el 'crypto winter' de 2018-2019?",
    # BLOQUE 2 — Patrones de ciclos y correlaciones
    "Basándote únicamente en el contexto proporcionado, ¿cuál es la relación histórica entre el halving de Bitcoin y el inicio de un bull market en Ethereum? ¿Cuántos meses de desfase suele observarse?",
    "¿Existe evidencia histórica de que Ethereum supere el rendimiento de Bitcoin durante la fase tardía de un ciclo alcista? Cita ciclos concretos para justificar tu respuesta.",
    "¿Qué es el ratio ETH/BTC y qué indica históricamente cuando ese ratio sube? ¿En qué ciclos ha marcado máximos?",
    "Describe las 4 fases clásicas de un ciclo de mercado cripto (acumulación, markup, distribución, markdown) y ejemplifícalas con fechas reales de BTC o ETH.",
    "¿Por qué algunos analistas sugieren que Ethereum opera en ciclos de 8 años mientras Bitcoin opera en ciclos de 4 años? ¿Qué datos históricos apoyan o refutan esa hipótesis?",
    # BLOQUE 3 — Razonamiento con incertidumbre
    "Con la información disponible, ¿puede afirmarse que el ciclo de 4 años de Bitcoin seguirá repitiéndose en el futuro? ¿Qué factores podrían romper ese patrón?",
    "¿Es posible predecir con precisión el precio máximo de Ethereum en un ciclo dado, basándose solo en datos de ciclos anteriores? Explica las limitaciones de ese enfoque.",
    "Si el contexto proporcionado no incluye datos sobre el precio actual de Bitcoin, ¿en qué fase del ciclo podría estar el mercado según indicadores on-chain históricos? Sé explícito sobre qué información te falta para responder con certeza.",
    "¿Qué métricas on-chain históricas han sido más útiles para identificar el techo de un ciclo alcista en Bitcoin? ¿Son igualmente válidas para Ethereum?",
    "¿Cuál es la diferencia estructural entre el bull market de 2017 y el de 2021 en términos de quién participó y cómo afectó eso a la volatilidad y la duración del ciclo?",
    # BLOQUE 4 — Trampas anti-alucinación
    "¿Cuál será el precio de Bitcoin en el próximo halving? Responde solo con información del contexto proporcionado, sin especular.",
    "¿Qué indicadores técnicos específicos señalaron exactamente el techo del ciclo de 2021 en Ethereum el día 10 de noviembre? Si no tienes esa información en el contexto, indícalo explícitamente.",
    "Según el contexto que tienes disponible, ¿existe algún patrón que permita predecir el momento exacto en que termina la fase de distribución de un ciclo? Si no hay evidencia suficiente, di por qué.",
    "¿Puede el comportamiento del precio de Ethereum en ciclos pasados utilizarse como modelo predictivo fiable para el siguiente ciclo? Lista explícitamente los supuestos que tendrías que asumir para que esa predicción sea válida.",
    "¿Qué información adicional necesitarías, más allá del contexto proporcionado, para hacer una predicción fundamentada sobre el próximo techo de ciclo de Bitcoin o Ethereum?",
]

BLOQUES = {
    "BLOQUE 1 — Hechos históricos verificables":  range(0, 5),
    "BLOQUE 2 — Patrones de ciclos y correlaciones": range(5, 10),
    "BLOQUE 3 — Razonamiento con incertidumbre":  range(10, 15),
    "BLOQUE 4 — Trampas anti-alucinación":         range(15, 20),
}

print(f"Eval set: {len(preguntas)} preguntas en {len(BLOQUES)} bloques")

Eval set: 20 preguntas en 4 bloques


In [29]:
resultados = []

for i, pregunta in enumerate(preguntas):
    bloque_actual = next(nombre for nombre, rango in BLOQUES.items() if i in rango)

    print(f"\n{'='*70}")
    print(f"[{bloque_actual}]")
    print(f"PREGUNTA {i+1}/{len(preguntas)}: {pregunta}")
    print("="*70)

    respuesta, log = responder_pregunta(pregunta, usar_rag=True)
    print(respuesta)

    resultados.append({
        "bloque":   bloque_actual,
        "numero":   i + 1,
        "pregunta": pregunta,
        "respuesta": respuesta,
        "log":      log,
    })

    if i < len(preguntas) - 1:
        print(f"\n⏳ Esperando {PAUSA_RATE_LIMIT}s para respetar rate limit...")
        time.sleep(PAUSA_RATE_LIMIT)

# Guardar resultados
os.makedirs(os.path.dirname(RUTA_RESULTADOS), exist_ok=True)
with open(RUTA_RESULTADOS, "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print(f"\n✓ Resultados guardados en: {RUTA_RESULTADOS}")
print(f"  Total preguntas evaluadas: {len(resultados)}")
print(f"  Errores: {sum(1 for r in resultados if r['respuesta'].startswith('❌'))}")


[BLOQUE 1 — Hechos históricos verificables]
PREGUNTA 1/20: ¿En qué año ocurrió el halving de Bitcoin de 2020 y cuánto bajó la recompensa de bloque? ¿Qué efecto histórico tuvo en el precio en los 12 meses siguientes?
❌ ERROR: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 8.500403596s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetri

KeyboardInterrupt: 